In [ ]:
import pandas as pd


In [ ]:
# Load data
df = pd.read_csv("01_data_raw/Brokers.csv")

# View first few rows
print(df.head())

  broker_id    broker_name      agency  experience_years  rating       city
0  BRK50000     Priya Shah  PropSquare               5.0     4.1      Nodia
1  BRK50001  Rohan Agarwal   CityHomes               5.0     3.8  Hyderabad
2  BRK50002   Kabir Sharma     PropHub               5.0     4.4  New Delhi
3  BRK50003    Sachin Nair   CityHomes               7.0     4.3     Surrat
4  BRK50004     Arjun Shah    HomeNest               3.0     3.5      Dehli


In [ ]:
# Check missing values
print(df.isnull().sum())

# Fill missing values
df['agency'] = df['agency'].fillna('Unknown')
df['city'] = df['city'].fillna('Not Specified')

# Drop duplicates if any
df = df.drop_duplicates(subset='broker_id', keep='first')


broker_id            0
broker_name          0
agency               0
experience_years    16
rating              25
city                 0
dtype: int64


In [ ]:
df[['experience_years', 'rating']].head(10)


,experience_years,rating
0,5.0,4.1
1,5.0,3.8
2,5.0,4.4
3,7.0,4.3
4,3.0,3.5
5,10.0,2.6
6,9.0,4.5
7,16.0,4.1
8,6.0,3.3
9,13.0,3.4


In [ ]:
df['experience_years'] = df['experience_years'].fillna(df['experience_years'].median())


In [ ]:
df['experience_years'] = df['experience_years'].fillna(0)


In [ ]:
# Convert data types if needed
df['experience_years'] = df['experience_years'].astype(float)
df['rating'] = df['rating'].astype(float)

In [ ]:
print(df.isnull().sum())


broker_id            0
broker_name          0
agency               0
experience_years     0
rating              25
city                 0
dtype: int64


In [ ]:
df['rating'] = df.groupby('city')['rating'].transform(lambda x: x.fillna(x.mean()))


In [ ]:
print(df.isnull().sum())

broker_id           0
broker_name         0
agency              0
experience_years    0
rating              0
city                0
dtype: int64


In [ ]:
# Remove brokers with negative or unrealistic experience
df = df[df['experience_years'] >= 0]

# Remove invalid ratings (assuming rating should be 1–5)
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

In [ ]:
# Remove extra spaces and standardize case
df['broker_name'] = df['broker_name'].str.strip().str.title()
df['agency'] = df['agency'].str.strip().str.title()
df['city'] = df['city'].str.strip().str.title()

In [ ]:
# Categorize brokers by experience
df['experience_level'] = pd.cut(df['experience_years'],
                                bins=[0, 2, 5, 10, 20, 50],
                                labels=['Beginner', 'Junior', 'Mid', 'Senior', 'Expert'])

# Average rating per city
city_avg = df.groupby('city')['rating'].mean().reset_index().rename(columns={'rating': 'city_avg_rating'})
df = df.merge(city_avg, on='city', how='left')


In [ ]:
# Label encoding or one-hot encoding
df = pd.get_dummies(df, columns=['city', 'agency'], drop_first=True)


In [ ]:
df.to_csv("brokers_cleaned.csv", index=False)
